In [26]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from datetime import datetime, timedelta
from IPython.display import display, clear_output
from sentinelhub import (
    SHConfig, SentinelHubRequest, DataCollection, 
    BBox, CRS, MimeType, bbox_to_dimensions
)

# 1. AUTH & CONFIG
config = SHConfig()
config.sh_client_id = "sh-1314cb54-dac7-46ca-869c-8c37fd193c7d"
config.sh_client_secret = "PjQ6GZVqZVHDLBKhdmI7jIMT7INDoVMo"
config.sh_base_url = "https://sh.dataspace.copernicus.eu"
config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"

# 2. AOI (Taranto)
bbox_coords = [17.082024, 40.346544, 17.356339, 40.507274]
aoi_bbox = BBox(bbox=bbox_coords, crs=CRS.WGS84)
image_size = bbox_to_dimensions(aoi_bbox, resolution=20)
cdse_collection = DataCollection['CDSE_TIMELINE']

# Updated Evalscript with SCL Cloud Masking
evalscript = """
//VERSION=3
function setup() {
  return {
    input: ["B03", "B04", "B05", "B06", "B07", "B08", "SCL", "dataMask"],
    output: { bands: 3, sampleType: "FLOAT32" }
  };
}

function evaluatePixel(s) {
  // 1. CLOUD & LAND MASKING (SCL)
  // We only want SCL value 6 (Water)
  // We exclude 3 (Shadow), 8 (Cloud Med), 9 (Cloud High), 10 (Cirrus)
  if (s.SCL !== 6 || s.dataMask === 0) return [0, 0, 0];

  // 2. ADDITIONAL WATER MASK (NDWI)
  // Double check to ensure we don't have shore pixels
  let ndwi = (s.B03 - s.B08) / (s.B03 + s.B08);
  if (ndwi < 0.1) return [0, 0, 0]; 

  // 3. SCIENTIFIC MATH
  let tss = (384.11 * s.B04) / (1 - (s.B04 / 0.1747));
  let re_reflectance = (s.B04 + s.B07) / 2;
  let s2rep = 705 + 35 * ((re_reflectance - s.B05) / (s.B06 - s.B05));
  
  return [tss, s2rep, 1.0];
}
"""

# 4. WEEKLY SCAN FETCH (Jan 2026 - Present)
history_data = {}
start_date = datetime(2025, 10, 1)
end_date = datetime.now()
current_date = start_date

print("Starting Weekly Time-Series Fetch for 2026...")

while current_date < end_date:
    week_end = current_date + timedelta(days=7)
    start_str = current_date.strftime("%Y-%m-%d")
    end_str = week_end.strftime("%Y-%m-%d")
    label = f"W: {start_str}"
    
    req = SentinelHubRequest(
        evalscript=evalscript,
        input_data=[SentinelHubRequest.input_data(
            data_collection=cdse_collection, 
            time_interval=(start_str, end_str), 
            mosaicking_order='leastCC'
        )],
        responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
        bbox=aoi_bbox, size=image_size, config=config
    )
    
    try:
        # We check for clouds by ensuring the response isn't just zeros
        data = req.get_data()[0]
        if np.any(data[:,:,2] > 0): # Check if any water pixels were found
            history_data[label] = {"tss": data[:,:,0], "chl": data[:,:,1]}
            print(f"✅ Success: {label}")
    except:
        pass # Skip weeks with full cloud cover
        
    current_date = week_end

# 5. DASHBOARD
output = widgets.Output()
date_slider = widgets.SelectionSlider(options=list(history_data.keys()), description='Week:')
metric_btn = widgets.ToggleButtons(options=[('Turbidity (TSS)', 'tss'), ('Chlorophyll (S2REP)', 'chl')], description='Layer:')

# Dynamic scaling based on algorithm physics
min_slider = widgets.FloatSlider(value=0.0, min=-5.0, max=710.0, step=0.1, description='Min:')
max_slider = widgets.FloatSlider(value=20.0, min=0.1, max=750.0, step=0.1, description='Max:')

def update_viz(change=None):
    with output:
        clear_output(wait=True)
        target = history_data[date_slider.value][metric_btn.value]
        
        # Mask land
        display_data = np.ma.masked_where(target <= 0, target)
        
        plt.figure(figsize=(14, 9))
        im = plt.imshow(display_data, 
                        vmin=min_slider.value, 
                        vmax=max_slider.value, 
                        cmap='magma' if metric_btn.value == 'tss' else 'viridis')
        
        plt.colorbar(im, label=f"Pixel Value ({metric_btn.value})")
        plt.title(f"Mar Piccolo Environmental Scan | {date_slider.value}")
        plt.axis('off')
        plt.show()

# Link everything
for w in [date_slider, metric_btn, min_slider, max_slider]:
    w.observe(update_viz, names='value')

# Preset scaling helper
def on_metric_change(change):
    if change['new'] == 'chl':
        min_slider.value, max_slider.value = 705.0, 725.0 # S2REP specific range
    else:
        min_slider.value, max_slider.value = 0.0, 15.0 # TSS specific range
metric_btn.observe(on_metric_change, names='value')

display(widgets.VBox([date_slider, metric_btn, widgets.HBox([min_slider, max_slider]), output]))
update_viz()

Starting Weekly Time-Series Fetch for 2026...
✅ Success: W: 2025-10-01
✅ Success: W: 2025-10-15
✅ Success: W: 2025-10-22
✅ Success: W: 2025-10-29
✅ Success: W: 2025-11-05
✅ Success: W: 2025-11-12
✅ Success: W: 2025-11-19
✅ Success: W: 2025-11-26
✅ Success: W: 2025-12-03
✅ Success: W: 2025-12-10
✅ Success: W: 2025-12-17
✅ Success: W: 2025-12-24
✅ Success: W: 2025-12-31
✅ Success: W: 2026-01-07
✅ Success: W: 2026-01-14
✅ Success: W: 2026-01-21
✅ Success: W: 2026-01-28
✅ Success: W: 2026-02-18
✅ Success: W: 2026-02-25
✅ Success: W: 2026-03-04
✅ Success: W: 2026-03-11
✅ Success: W: 2026-03-18
✅ Success: W: 2026-03-25
✅ Success: W: 2026-04-01
✅ Success: W: 2026-04-08
✅ Success: W: 2026-04-15
✅ Success: W: 2026-04-22
